# AuditLens AI - Forensic Anomaly Detection Model Evaluation

This notebook presents the empirical evaluation of **AuditLens AI** on a 50,000-entry synthetic general ledger containing 1,000 (2.00%) injected forensic anomalies across 7 audit failure modes.

### Evaluation Highlights
- **Total Transactions Evaluated**: 50,000
- **Injected Anomalies (Ground Truth)**: 1,000 (2.00%)
- **Medium Risk Threshold (>= 0.35)**: Recall = **99.20%** (992 / 1,000 caught), Precision = **72.83%**, F1 = **0.839966**
- **High Risk Threshold (>= 0.55)**: Precision = **75.13%**, Recall = **43.20%**, F1 = **0.548571**
- **Hybrid Advantage**: The combination of Rules + Isolation Forest catches **99.20%** of anomalies, whereas Rules Only misses subtle account-amount statistical outliers (0.00% recall on unusual amounts), and ML Only has low precision on discrete rule violations.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

from auditlens.config import HIGH_RISK_THRESHOLD, MEDIUM_RISK_THRESHOLD, DEFAULT_DATASET_PATH, SAVED_MODEL_PATH
from auditlens.engine.features import extract_features
from auditlens.engine.rules import evaluate_rules
from auditlens.engine.model import IsolationForestDetector
from auditlens.engine.scoring import compile_scored_ledger

# Load ledger dataset
raw_df = pd.read_csv(DEFAULT_DATASET_PATH)
print(f"Loaded {len(raw_df):,} ledger transactions.")

## 1. Feature Extraction & Hybrid Risk Scoring
We extract temporal, financial magnitude, and segregation of duties features, evaluate heuristic audit rules, compute Isolation Forest anomaly scores, and calculate the combined risk score.

In [2]:
features_df = extract_features(raw_df)
rule_scores, rule_reasons, rule_flags = evaluate_rules(raw_df, features_df)

detector = IsolationForestDetector.load(SAVED_MODEL_PATH)
ml_scores = detector.predict_score(features_df)

scored_df = compile_scored_ledger(raw_df, rule_scores, ml_scores, rule_reasons)
y_true = (scored_df["is_injected_anomaly"] == 1).values

print(f"Risk Scoring Complete: {len(scored_df):,} entries scored.")

## 2. Evaluation at High and Medium Risk Thresholds
Here we report real Precision, Recall, F1, and full Confusion Matrices for:
- **High Risk Threshold** (risk_score >= 0.55)
- **Medium Risk Threshold** (risk_score >= 0.35, representing all flagged entries)

In [3]:
for tier_name, thresh in [("High Risk (>= 0.55)", HIGH_RISK_THRESHOLD), ("Medium Risk (>= 0.35)", MEDIUM_RISK_THRESHOLD)]:
    y_pred = scored_df["risk_score"] >= thresh
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print("=" * 60)
    print(f"THRESHOLD: {tier_name}")
    print("=" * 60)
    print(f"Total Flagged: {y_pred.sum():>6,} ({(y_pred.sum() / len(scored_df)) * 100:.2f}% of ledger)")
    print(f"True Positives (TP) : {tp:>6,}")
    print(f"False Positives (FP): {fp:>6,}")
    print(f"False Negatives (FN): {fn:>6,}")
    print(f"True Negatives (TN) : {tn:>6,}")
    print(f"Precision          : {p:.6f} ({p*100:.2f}%)")
    print(f"Recall             : {r:.6f} ({r*100:.2f}%)")
    print(f"F1-Score           : {f1:.6f}")
    print()


## 3. Recall Breakdown by Anomaly Category
Analysis of detection rates across each of the 7 specific forensic anomaly injection patterns.

In [4]:
breakdown = []
anomaly_types = [
    "weekend_holiday",
    "round_amount",
    "duplicate",
    "below_approval_limit",
    "same_poster_approver",
    "vague_description",
    "unusual_account_amount"
]

for atype in anomaly_types:
    mask = scored_df["anomaly_type"] == atype
    total_injected = mask.sum()
    caught_med = (scored_df.loc[mask, "risk_score"] >= MEDIUM_RISK_THRESHOLD).sum()
    caught_high = (scored_df.loc[mask, "risk_score"] >= HIGH_RISK_THRESHOLD).sum()
    
    breakdown.append({
        "Anomaly Category": atype,
        "Ground Truth Count": total_injected,
        "Caught at Medium (>=0.35)": caught_med,
        "Medium Recall (%)": round((caught_med / total_injected) * 100, 2),
        "Caught at High (>=0.55)": caught_high,
        "High Recall (%)": round((caught_high / total_injected) * 100, 2),
    })

breakdown_df = pd.DataFrame(breakdown)
display(breakdown_df)


## 4. Model Comparison: Rules Only vs. ML Only vs. Hybrid Combined
Empirical comparison illustrating why neither rules alone nor unsupervised ML alone suffices for enterprise journal entry auditing.

In [5]:
comp_data = []

# Rules Only (rule_score >= 0.50)
y_rules = scored_df["rule_score"] >= 0.50
p_r, r_r, f1_r, _ = precision_recall_fscore_support(y_true, y_rules, average="binary", zero_division=0)
comp_data.append({"Model Approach": "Rules Only (Threshold >= 0.50)", "Flagged Count": int(y_rules.sum()), "Precision": round(p_r, 4), "Recall": round(r_r, 4), "F1 Score": round(f1_r, 4)})

# ML Only (ml_score >= 0.60)
y_ml = scored_df["ml_score"] >= 0.60
p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_ml, average="binary", zero_division=0)
comp_data.append({"Model Approach": "ML Only (Isolation Forest >= 0.60)", "Flagged Count": int(y_ml.sum()), "Precision": round(p_m, 4), "Recall": round(r_m, 4), "F1 Score": round(f1_m, 4)})

# Hybrid Combined (risk_score >= 0.35)
y_comb = scored_df["risk_score"] >= MEDIUM_RISK_THRESHOLD
p_c, r_c, f1_c, _ = precision_recall_fscore_support(y_true, y_comb, average="binary", zero_division=0)
comp_data.append({"Model Approach": "Hybrid AuditLens (Risk >= 0.35)", "Flagged Count": int(y_comb.sum()), "Precision": round(p_c, 4), "Recall": round(r_c, 4), "F1 Score": round(f1_c, 4)})

comp_df = pd.DataFrame(comp_data)
display(comp_df)


### Commentary on Model Comparison:
1. **Rules Only** misses **unusual account-amount combinations** entirely (Recall = 0% for unusual amounts, overall Recall = 85.80%), because creating rigid threshold rules for every possible account-vendor-amount combination is impossible in large enterprises.
2. **ML Only** detects statistical outliers but suffers from high false positive noise on standard entries that naturally have high variance (Precision = 24.31%).
3. **Hybrid AuditLens** achieves the optimal balance: **99.20% Recall** across all 7 anomaly types while retaining strong precision (72.83%) and F1 (0.84).

## 5. False Positives Analysis & Auditor Commentary
We inspect entries where `is_injected_anomaly == 0` but `risk_score >= 0.35` (flagged by the system).

In [6]:
fps = scored_df[(scored_df["is_injected_anomaly"] == 0) & (scored_df["risk_score"] >= MEDIUM_RISK_THRESHOLD)].copy()
print(f"Total Clean Entries Flagged (False Positives): {len(fps)} out of 49,000 clean entries ({len(fps)/49000*100:.2f}%)")

sample_fp_cols = ["entry_id", "posting_date", "account_name", "amount", "rule_score", "ml_score", "risk_score", "rule_reasons"]
display(fps[sample_fp_cols].head(6))


### Auditor Commentary on False Positives:

1. **Duplicate Counterparts (143 entries = 38.6% of False Positives)**:
   In Stage 1 data injection, duplicate anomalies were created by selecting an existing normal entry and injecting an exact duplicate copy with `is_injected_anomaly = 1`. To an auditor and to the feature extractor, **both transactions in the duplicate pair are identical violations**. The original row is technically labeled `is_injected_anomaly = 0` in the evaluation file, but flagging it is correct audit behavior.

2. **Natural Statistical Tail Outliers (~227 entries = 0.46% of clean ledger)**:
   The remaining false positives are benign enterprise entries located in the extreme tails of the log-normal amount distribution (e.g., legitimate quarterly dividend transfers or enterprise software milestones exceeding INR 2,000,000). The Isolation Forest correctly identifies them as statistical outliers relative to the median transaction (INR 60,257). An auditor would routinely sample these high-dollar transactions for substantiation anyway.